# Từ GPT-2 tới Llama: ablation từng thành phần, đo KV cache

Bạn ghép các mảnh của bài Basic 02 (RMSNorm, RoPE, SwiGLU, GQA, KV cache) thành một `ModernGPT` có cờ bật/tắt từng thành
phần, rồi trả lời bằng số: mỗi thay đổi từ GPT-2 sang Llama đóng góp bao nhiêu bpc ở quy mô nhỏ, chênh lệch nào thật và
chênh lệch nào chỉ là nhiễu giữa hai seed, KV cache nhanh hơn bao nhiêu và tốn bao nhiêu bộ nhớ.

**Cách làm bài**
- Lớp `ModernConfig`, `ModernGPT` được test chấm (đặc tả ở mục 1). Thí nghiệm tự thiết kế.
- Cell thứ ba nạp `modern.py` của bạn (bài Basic 02); bài đó chưa xong thì nạp lời giải.
- GPU: chạy bằng kernel `ai-gpu`; cấu hình ở README của bài.
- Chấm: `uv run pytest tracks/ai-engineer/modules/05-transformers-llm-architecture/exercises/advanced/02-modern-llm-ablation -q`

In [ ]:
# explore: giới hạn luồng
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")

In [ ]:
import importlib.util
import math
import sys
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
MODULE = ROOT / "tracks/ai-engineer/modules/05-transformers-llm-architecture"


def load_modern():
    for path in [MODULE / "exercises/basic/02-rope-gqa-kv-cache/starter/modern.py",
                 MODULE / "solutions/basic/02-rope-gqa-kv-cache/modern.py"]:
        spec = importlib.util.spec_from_file_location("modern_parts", path)
        module = importlib.util.module_from_spec(spec)
        sys.modules["modern_parts"] = module
        spec.loader.exec_module(module)
        try:
            module.GQAttention(8, 2, 1, 4)(torch.zeros(1, 2, 8))
            module.RMSNorm(4)(torch.zeros(1, 4))
            module.SwiGLU(4, 8)(torch.zeros(1, 4))
            return module
        except NotImplementedError:
            continue
    raise RuntimeError("không nạp được modern.py")


modern = load_modern()
RMSNorm, SwiGLU, GQAttention = modern.RMSNorm, modern.SwiGLU, modern.GQAttention

## 1. `ModernGPT`

`ModernConfig` là dataclass với các trường (đúng tên, đúng mặc định): `vocab_size=8000`, `block_size=128`, `n_layer=4`,
`n_head=4`, `n_kv_head=2`, `n_embd=256`, `ffn_hidden=704`, `pos="rope"`, `norm="rms"`, `mlp="swiglu"`.

`ModernGPT(config)`:

- `tok_emb` (`nn.Embedding`); nếu `pos == "learned"` thêm `pos_emb` (`nn.Embedding(block_size, n_embd)`), ngược lại
  `pos_emb = None`.
- `blocks`: `n_layer` khối pre-norm, mỗi khối có `norm1`, `attn` (`GQAttention(n_embd, n_head, n_kv_head, block_size)`),
  `norm2`, `mlp`. Norm là `RMSNorm(n_embd)` khi `norm == "rms"`, `nn.LayerNorm(n_embd)` khi `"layer"`. MLP là
  `SwiGLU(n_embd, ffn_hidden)` khi `"swiglu"`; khi `"gelu"` là `fc` (`n_embd → 4·n_embd`) → GELU → `w2` (`4·n_embd → n_embd`),
  cả hai không bias. Khi `pos == "learned"`, tắt RoPE bằng cách đặt buffer `cos` của attention thành 1 và `sin` thành 0
  (phép quay đồng nhất).
- `norm` cuối (cùng loại với các khối), `lm_head` không bias, **dùng chung** trọng số với `tok_emb`.
- `forward(idx, targets=None, caches=None) -> (logits, loss, new_caches)`: `caches` là `None` hoặc list `n_layer` phần tử
  (mỗi phần tử `None` hoặc cache `(k, v)` của `GQAttention`). Token mới có vị trí bắt đầu từ số token đã có trong cache (dùng
  cho cả `pos_emb`). `loss` là cross-entropy trung bình khi có `targets`, ngược lại `None`.
- `generate(idx, max_new_tokens, temperature=1.0, top_k=None, generator=None, use_cache=True)`: chạy `eval`, không gradient,
  quy ước temperature/top-k giống bài Basic 01. Có cache: prefill cả `idx` một lần rồi mỗi bước chỉ đưa **một** token mới. Không
  cache: mỗi bước forward lại `idx[:, -block_size:]`. Cả hai cách cho đúng cùng token với greedy (khi tổng độ dài không vượt
  `block_size`).

In [ ]:
@dataclass
class ModernConfig:
    # TODO: các trường và mặc định theo đặc tả
    pass


class ModernGPT(nn.Module):
    def __init__(self, config: ModernConfig):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None, caches: list | None = None):
        # TODO
        raise NotImplementedError

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int, temperature: float = 1.0, top_k: int | None = None,
                 generator: torch.Generator | None = None, use_cache: bool = True) -> torch.Tensor:
        # TODO
        raise NotImplementedError

## 2. Gợi ý bố cục thí nghiệm (không chấm)

1. Dữ liệu và đánh giá đúng như bài Intermediate 01 (1 000 bài, BPE 8 000, `evaluate_nll` cửa sổ trượt, bpc).
2. Baseline: `GPT` bài Basic 01 (GPT-2: LayerNorm, GELU, position embedding học được, MHA). Bản đầy đủ: `ModernGPT` mặc định.
3. Ablation: từ bản đầy đủ đổi **một** thứ mỗi lần (`pos="learned"`, `norm="layer"`, `mlp="gelu"`, `n_kv_head` = 4 và 1),
   cùng số bước, batch, learning rate; ghi số tham số. Chạy lại ít nhất hai cấu hình với seed khác để biết độ nhiễu.
4. KV cache: thời gian sinh 64/256/768 token có và không cache; bộ nhớ cache của model thật bằng `kv_cache_bytes`.
5. Viết `REPORT.md`.

In [ ]:
# explore: TODO: thí nghiệm của bạn